In [1]:
import subprocess, sys

# 1. Install Python packages
packages = [
    "numpy<2.0",
    "nemo_toolkit[asr]",
    "soundfile",
    "huggingface_hub",
    "hf-transfer",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages, check=True)

# 2. Update and install system libraries using sudo
# We add 'sudo' and 'apt-get update' to ensure the package list is fresh
try:
    print("Updating package list...")
    subprocess.run(["sudo", "apt-get", "update", "-q"], check=True)
    
    print("Installing system dependencies...")
    subprocess.run(["sudo", "apt-get", "install", "-y", "-q", "libsndfile1", "ffmpeg"], check=True)
    
    print("✅ Installation complete. RESTART SESSION NOW.")
except subprocess.CalledProcessError as e:
    print(f"❌ Installation failed with error: {e}")

Updating package list...
Get:1 https://nvidia.github.io/libnvidia-container/stable/deb/amd64  InRelease [1477 B]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:3 https://packages.cloud.google.com/apt cloud-sdk InRelease
Hit:4 https://download.docker.com/linux/ubuntu noble InRelease
Get:5 https://cli.github.com/packages stable InRelease [3917 B]
Hit:6 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble InRelease
Hit:7 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:8 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:9 https://archive.ubuntu.com/ubuntu noble InRelease
Hit:10 http://deb.wakemeops.com/wakemeops stable InRelease
Hit:11 https://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:12 https://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:13 https://cloud.archive.ubuntu.com/ubuntu noble InRelease
Hit:14 https://security.ubuntu.com/ubuntu noble-security InRelease

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.



Building dependency tree...
Reading state information...
libsndfile1 is already the newest version (1.2.2-1ubuntu5.24.04.1).
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 46 not upgraded.
✅ Installation complete. RESTART SESSION NOW.


In [2]:
import os

# Lightning AI — set this in your Studio's Environment Variables panel
HF_TOKEN = ""

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"   # fast Rust-based downloads

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not set! Add it in Lightning AI → Studio → Environment Variables")

print("✅ Token ready")

✅ Token ready


In [3]:
MODEL_REPO_ID   = "hishab/titu_stt_bn_conformer_large"
DATASET_REPO_ID = "Sanjidh090/Lipi-Ghor-bn-882-SSTT"
OUTPUT_REPO_ID  = "hasans090/titu_conformer_inference_lp"

DATASET_AUDIO_FOLDER = "data"
LOCAL_AUDIO_CACHE    = "/tmp/audio_cache"

VERSION           = 2      # ← change before each run: 1, 2, 3, 4, 5
FILES_PER_VERSION = 204    # 204 × 5 = 1020, covers all 1019

OUTPUT_CSV = f"/tmp/titu_conformer_lipighor_v{VERSION}.csv"

CHUNK_SEC   = 20
OVERLAP_SEC = 2
BATCH_SIZE  = 64   # RTX 6000 has 24GB — conformer is lightweight, push it

import os
os.makedirs(LOCAL_AUDIO_CACHE, exist_ok=True)
print(f"✅ Config ready — v{VERSION}, files {(VERSION-1)*FILES_PER_VERSION+1} to {VERSION*FILES_PER_VERSION}")

✅ Config ready — v2, files 205 to 408


In [4]:
from huggingface_hub import hf_hub_download
import nemo.collections.asr as nemo_asr
from omegaconf import OmegaConf
import torch

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print(f"\nDownloading model: {MODEL_REPO_ID}")
nemo_path = hf_hub_download(
    repo_id  = MODEL_REPO_ID,
    filename = "titu_stt_bn_conformer_large.nemo",
    token    = HF_TOKEN,
)

print("Loading model on GPU...")
model = nemo_asr.models.ASRModel.restore_from(nemo_path)
model = model.to("cuda")
model.eval()

# Greedy batch decoding — fastest for inference
model.change_decoding_strategy(OmegaConf.create({"strategy": "greedy_batch"}))

print("✅ Model loaded and ready")

[NeMo W 2026-04-13 18:31:53 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 102.0 GB

Loading model on GPU...
[NeMo I 2026-04-13 18:31:54 mixins:184] Tokenizer SentencePieceTokenizer initialized with 128 tokens


[NeMo W 2026-04-13 18:31:54 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /workspace/datasets/bangla-open-asr-data/bangla_open_asr_manifests/non-telephony-all-train-val.json
    sample_rate: 16000
    batch_size: 32
    shuffle: true
    num_workers: 8
    pin_memory: true
    use_start_end_token: true
    trim_silence: false
    max_duration: 30.0
    min_duration: 0.1
    is_tarred: false
    tarred_audio_filepaths:
    - - /data2/nemo_asr/nemo_asr_set_3.0/bucket1/audio__OP_0..8191_CL_.tar
    - - /data2/nemo_asr/nemo_asr_set_3.0/bucket2/audio__OP_0..8191_CL_.tar
    - - /data2/nemo_asr/nemo_asr_set_3.0/bucket3/audio__OP_0..8191_CL_.tar
    - - /data2/nemo_asr/nemo_asr_set_3.0/bucket4/audio__OP_0..8191_CL_.tar
    - - /data2/nemo_asr/nemo_asr_set_3.0/bucket5/audio__OP_0..8191_CL_.tar
    - - /data2/nemo_asr/

[NeMo I 2026-04-13 18:31:55 save_restore_connector:285] Model EncDecCTCModelBPE was successfully restored from /teamspace/studios/this_studio/.cache/huggingface/hub/models--hishab--titu_stt_bn_conformer_large/snapshots/42041012a7cdb9df071eac4cef33b8a6182eb687/titu_stt_bn_conformer_large.nemo.
[NeMo I 2026-04-13 18:31:55 ctc_bpe_models:357] Changed decoding strategy to 
    strategy: greedy_batch
    preserve_alignments: null
    compute_timestamps: null
    word_seperator: ' '
    segment_seperators:
    - .
    - '!'
    - '?'
    segment_gap_threshold: null
    ctc_timestamp_type: all
    batch_dim_index: 0
    greedy:
      preserve_alignments: false
      compute_timestamps: false
      preserve_frame_confidence: false
      confidence_method_cfg:
        name: entropy
        entropy_type: tsallis
        alpha: 0.33
        entropy_norm: exp
        temperature: DEPRECATED
      ngram_lm_model: null
      ngram_lm_alpha: 0.0
      boosting_tree:
        model_path: null
        k

In [5]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
api.create_repo(
    repo_id   = OUTPUT_REPO_ID,
    repo_type = "dataset",
    private   = True,
    exist_ok  = True,
)
print(f"✅ Repo ready: https://huggingface.co/datasets/{OUTPUT_REPO_ID}")

✅ Repo ready: https://huggingface.co/datasets/hasans090/titu_conformer_inference_lp


In [6]:
import os, tempfile, subprocess, time
from pathlib import Path
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download, list_repo_files, upload_file

def get_duration(path):
    result = subprocess.run([
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(path)
    ], capture_output=True, text=True)
    return float(result.stdout.strip())

def extract_chunk_ffmpeg(audio_path, start_sec, duration_sec, target_sr=16000):
    tmp = tempfile.mktemp(suffix=".wav")
    subprocess.run([
        "ffmpeg", "-y",
        "-ss", str(start_sec),
        "-t",  str(duration_sec),
        "-i",  str(audio_path),
        "-ar", str(target_sr),
        "-ac", "1", tmp
    ], capture_output=True, check=True)
    return tmp

def dedup_overlap(t1, t2, max_w=8):
    w1, w2 = t1.split(), t2.split()
    for n in range(min(max_w, len(w1), len(w2)), 0, -1):
        if w1[-n:] == w2[:n]:
            return n
    return 0

def extract_text(hyp_item):
    if isinstance(hyp_item, str):        return hyp_item
    elif hasattr(hyp_item, "text"):      return hyp_item.text
    elif hasattr(hyp_item, "__iter__"):  return hyp_item[0]
    return str(hyp_item)

def list_audio_files_on_hf(dataset_repo, folder, token):
    AUDIO_EXTS = {".wav", ".mp3", ".flac", ".m4a", ".ogg"}
    all_files  = list_repo_files(dataset_repo, repo_type="dataset", token=token)
    return sorted([
        f for f in all_files
        if f.startswith(folder + "/") and Path(f).suffix.lower() in AUDIO_EXTS
    ])

def download_single_audio(dataset_repo, hf_path, local_dir, token, max_retries=3):
    for attempt in range(max_retries):
        try:
            return hf_hub_download(
                repo_id   = dataset_repo,
                filename  = hf_path,
                repo_type = "dataset",
                token     = token,
                local_dir = local_dir,
            )
        except Exception as e:
            print(f"⚠️ Download attempt {attempt+1}/{max_retries} failed. Retrying...")
            if attempt < max_retries - 1:
                time.sleep(3)
            else:
                raise e

def push_csv_to_hf(local_csv, output_repo, token):
    upload_file(
        path_or_fileobj = local_csv,
        path_in_repo    = Path(local_csv).name,
        repo_id         = output_repo,
        repo_type       = "dataset",
        token           = token,
    )

def load_existing_csv_from_hf(output_repo, csv_filename, token):
    try:
        path = hf_hub_download(
            repo_id        = output_repo,
            filename       = csv_filename,
            repo_type      = "dataset",
            token          = token,
            force_download = True,
        )
        df = pd.read_csv(path)
        print(f"✅ Loaded from HF — {len(df)} rows done so far")
        return df
    except Exception as e:
        print(f"ℹ️ No existing CSV (starting fresh): {e}")
        return None

def transcribe_chunks(tmp_files, batch_size):
    """Single GPU — feed all chunks in batches directly."""
    all_texts = []
    for i in range(0, len(tmp_files), batch_size):
        batch = tmp_files[i : i + batch_size]
        with torch.no_grad():
            hyps = model.transcribe(batch, batch_size=len(batch))
        all_texts.extend([extract_text(h) for h in hyps])
    return all_texts

print("✅ Utilities ready")

✅ Utilities ready


In [7]:
hf_audio_paths = list_audio_files_on_hf(DATASET_REPO_ID, DATASET_AUDIO_FOLDER, HF_TOKEN)
TOTAL_ALL = len(hf_audio_paths)

start_idx = (VERSION - 1) * FILES_PER_VERSION
end_idx   = min(VERSION * FILES_PER_VERSION, TOTAL_ALL)
my_files  = hf_audio_paths[start_idx:end_idx]
TOTAL     = len(my_files)

print(f"Total files in repo : {TOTAL_ALL}")
print(f"This version (v{VERSION}): files [{start_idx}:{end_idx}] — {TOTAL} files\n")

# ── Resume ────────────────────────────────────────────────────────────────────
hf_df = load_existing_csv_from_hf(OUTPUT_REPO_ID, Path(OUTPUT_CSV).name, HF_TOKEN)

if hf_df is not None:
    done_ids = set(hf_df["id"].astype(str).tolist())
    results  = hf_df.to_dict("records")
    hf_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
elif Path(OUTPUT_CSV).exists():
    done_df  = pd.read_csv(OUTPUT_CSV)
    done_ids = set(done_df["id"].astype(str).tolist())
    results  = done_df.to_dict("records")
    print(f"Resuming from local — {len(done_ids)} done")
else:
    done_ids = set()
    results  = []
    print("Starting fresh")

print(f"\n📊 {len(done_ids)} done / {TOTAL} total — {TOTAL - len(done_ids)} remaining\n")

# ── Loop ──────────────────────────────────────────────────────────────────────
for i, hf_path in enumerate(my_files):
    file_id = Path(hf_path).stem

    if file_id in done_ids:
        continue

    os.system("clear")
    print(f"{'─'*60}")
    print(f"  [v{VERSION} — {len(results)+1}/{TOTAL}]  {file_id}")
    print(f"{'─'*60}\n")

    try:
        print("⬇  Downloading...")
        audio_path = download_single_audio(DATASET_REPO_ID, hf_path, LOCAL_AUDIO_CACHE, HF_TOKEN)

        total_sec = get_duration(audio_path)
        print(f"⏱  Duration: {total_sec:.0f}s ({total_sec/60:.1f} min)")

        step = CHUNK_SEC - OVERLAP_SEC
        starts, pos = [], 0.0
        while pos < total_sec:
            starts.append(pos)
            if pos + CHUNK_SEC >= total_sec: break
            pos += step
        print(f"🔪  Chunks: {len(starts)}\n")

        tmp_files = []
        for start in starts:
            dur = min(CHUNK_SEC, total_sec - start)
            tmp_files.append(extract_chunk_ffmpeg(audio_path, start, dur))

        chunk_texts = transcribe_chunks(tmp_files, BATCH_SIZE)

        for tmp in tmp_files:
            os.remove(tmp)

        for j, text in enumerate(chunk_texts):
            print(f"  chunk {j+1:>3}/{len(starts)}: {text[:70]}")

        words = chunk_texts[0].split() if chunk_texts else []
        for k in range(1, len(chunk_texts)):
            skip = dedup_overlap(chunk_texts[k-1], chunk_texts[k]) if OVERLAP_SEC > 0 else 0
            words.extend(chunk_texts[k].split()[skip:])
        transcript = " ".join(words).strip()

        os.remove(audio_path)
        print(f"\n✅  {len(transcript.split())} words total")

    except Exception as ex:
        transcript = ""
        print(f"\n❌  ERROR: {ex}")

    results.append({"id": file_id, "transcript": transcript})
    pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

    files_done = len(results)
    is_last    = (i == TOTAL - 1)
    if files_done % 10 == 0 or is_last:
        try:
            push_csv_to_hf(OUTPUT_CSV, OUTPUT_REPO_ID, HF_TOKEN)
            print(f"📤  CSV pushed to HF ({files_done} rows)")
        except Exception as e:
            print(f"⚠️  Push failed: {e}")

print(f"\n✅ Version {VERSION} done — {len(results)} rows")

Total files in repo : 1019
This version (v2): files [204:408] — 204 files



titu_conformer_lipighor_v2.csv:   0%|          | 0.00/19.5M [00:00<?, ?B/s]

✅ Loaded from HF — 204 rows done so far

📊 204 done / 204 total — 0 remaining


✅ Version 2 done — 204 rows
